# 05 — Teacher vs Student LLM Evaluation

This notebook compares **teacher** and **student** LLM outputs captured by the bounded LLM assistant. It relies on two ledger files populated by the orchestrator + `llm_assistant`:

- `audit/ledger/decision_cards.jsonl` — one record per triage decision, including `llm_tier`, `llm_provider`, `llm_model`, and the served `analyst_summary` / `next_steps`.
- `audit/ledger/teacher_shadow.jsonl` — teacher outputs recorded when the assistant runs in `teacher_shadow` or `teacher_only` mode.

Use this notebook to:

1. Measure the teacher/student/fallback mix currently served.
2. Compare served outputs to teacher shadows on matching `event_id`s.
3. Audit fallback rates per language group (equity check).

> The notebook makes only read calls. It never edits the ledger.

In [ ]:
import json
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd

LEDGER = Path('/workspace/data/ledger')
# In the default compose layout the ledger lives outside the notebook bind mount.
# Fall back to relative paths if the above does not exist.
CANDIDATES = [
    LEDGER,
    Path('/workspace/ledger'),
    Path('/app/ledger'),
    Path('../../audit/ledger').resolve(),
]
for c in CANDIDATES:
    if c.exists():
        LEDGER = c
        break

print('Using ledger dir:', LEDGER)
print('Exists:', LEDGER.exists())

In [ ]:
def read_jsonl(path):
    rows = []
    if not path.exists():
        return rows
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except Exception:
                continue
    return rows

decisions = read_jsonl(LEDGER / 'decision_cards.jsonl')
shadows   = read_jsonl(LEDGER / 'teacher_shadow.jsonl')

print(f'Loaded {len(decisions)} decision cards, {len(shadows)} teacher shadows')

## 1. Served tier mix

What fraction of realtime requests were served by the student vs teacher vs fallback? A high fallback rate is a signal that the student model is misconfigured or unreachable.

In [ ]:
df = pd.DataFrame(decisions)
if not df.empty:
    tier_counts = df['llm_tier'].fillna('unknown').value_counts()
    provider_counts = df['llm_provider'].fillna('unknown').value_counts()
    display(tier_counts.to_frame('count'))
    display(provider_counts.to_frame('count'))
else:
    print('No decisions logged yet. Run a scenario from the simulator first.')

## 2. Fallback rate by language

If one language group hits fallback more often than others, the student model is failing that group (OOV tokens, JSON-schema compliance) and should not be promoted over the teacher.

In [ ]:
if not df.empty:
    df['is_fallback'] = (df['llm_tier'].fillna('unknown') == 'fallback').astype(int)
    rate = df.groupby('language')['is_fallback'].agg(['count', 'sum', 'mean']).rename(
        columns={'count': 'events', 'sum': 'fallbacks', 'mean': 'fallback_rate'})
    display(rate.sort_values('fallback_rate', ascending=False))

## 3. Served vs Teacher agreement

For events where both a served decision card and a teacher shadow output exist, how often do they agree on structure and substance?

These metrics are **deliberately simple**. For policy decisions use richer metrics (human grading, semantic similarity) in a follow-up notebook.

In [ ]:
shadow_by_event = {}
for s in shadows:
    ev = (s.get('request') or {}).get('event_id')
    if ev:
        shadow_by_event[ev] = s

rows = []
for d in decisions:
    ev = d.get('event_id')
    s = shadow_by_event.get(ev)
    if not s:
        continue
    t_out = s.get('teacher_output') or {}
    served_steps = d.get('next_steps') or []
    teacher_steps = t_out.get('next_steps') or []
    served_sum = (d.get('analyst_summary') or '').lower()
    teacher_sum = (t_out.get('analyst_summary') or '').lower()
    served_words = set(w for w in served_sum.split() if len(w) > 4)
    teacher_words = set(w for w in teacher_sum.split() if len(w) > 4)
    overlap = len(served_words & teacher_words)
    rows.append({
        'event_id': ev,
        'language': d.get('language'),
        'served_tier': d.get('llm_tier'),
        'steps_match_len': int(len(served_steps) == len(teacher_steps)),
        'summary_overlap_words': overlap,
    })

cmp = pd.DataFrame(rows)
print(f'Compared {len(cmp)} paired events')
display(cmp.head())
if not cmp.empty:
    display(cmp.groupby('language')[['steps_match_len', 'summary_overlap_words']].mean())

## 4. Decision rule

Before promoting the student to handle events without teacher shadow:

- `fallback_rate` must be below 5% in every language group.
- `steps_match_len` must average above 0.95.
- A human reviewer must sign off via an entry in `docs/overrides_log.md`.

Document the decision in `docs/threshold_changes.md`.